### Data Shuffle, Skew Data, Disk Spilling & AQE - Demo
**Data Shuffle:** Ocurre cuando Spark necesita redistribuir los datos entre diferentes particiones y ejecutores del clúster para realizar determinadas operaciones.

**Data Skew:** Ocurre cuando los datos se distribuyen de manera desigual entre las particiones, provocando que algunas contengan muchos más registros que otras.

**Disk Spilling:** Ocurre cuando la memoria es insuficiente, Spark puede escribir temporalmente datos intermedios en disco para continuar con el procesamiento.

**Adaptive Query Execution(AQE):** consiste en la reoptimización de consultas que tiene lugar durante su ejecución, basándose en estadísticas obtenidas en tiempo de ejecución.<br>
En Spark 3.0, AQE incluye tres funcionalidades principales:

- Dynamically coalescing shuffle partitions (Combinación dinámica de particiones Shuffle).
- Dynamically optimizing skew joins (Optimización dinámica de joins con asimetría de datos).
- Dynamically switching join strategies (Cambio dinámico de las estrategias de JOIN).


#### Paso 1: Usar el catalog "demo"

In [0]:
%sql
USE CATALOG demo;

#### Paso 2: Crear el catalog "performance_optimization" y las tablas "items" y "sales"

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS performance_optimization;
USE performance_optimization;
 
DROP TABLE IF EXISTS items;
DROP TABLE IF EXISTS sales;
 
-- Crear la tabla"items".
CREATE TABLE items
AS
SELECT id AS i_item_id,
CAST(rand() * 1000 AS INT) AS i_price
FROM RANGE(30000000);
 
-- Crear la tabla "sales" con asimetría de datos.
-- El Item con "id 100" está presente en el 80 % de todas las ventas.
CREATE TABLE sales
AS
SELECT CASE WHEN rand() < 0.8 THEN 100 ELSE CAST(rand() * 30000000 AS INT) END AS s_item_id,
CAST(rand() * 100 AS INT) AS s_quantity,
DATE_ADD(current_date(), - CAST(rand() * 360 AS INT)) AS s_date
FROM RANGE(1000000000);

#### Paso 3: Utilizar las tablas "items" y "sales" para la demostración

##### 3.1. Desactivar AQE y comprobar Data Shuffle, Skew Data y Disk Spilling

In [0]:
spark.conf.get("spark.sql.adaptive.enabled")

In [0]:
spark.conf.set("spark.sql.adaptive.enabled", "false")

In [0]:
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "false")

In [0]:
spark.conf.get("spark.sql.adaptive.skewJoin.enabled")

In [0]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

In [0]:
spark.conf.get("spark.sql.autoBroadcastJoinThreshold")

In [0]:
spark.conf.get("spark.sql.shuffle.partitions")

In [0]:
# Obtener el importe total de las ventas, agrupado por fecha de venta, para artículos con un precio inferior a 10.
skew_df = spark.sql("""SELECT s_date, sum(s_quantity * i_price) AS total_sales
FROM sales
JOIN items ON s_item_id = i_item_id
WHERE i_price < 10
GROUP BY s_date
ORDER BY total_sales DESC;""") 

In [0]:
# El análisis estático (explain) muestra el plan inicial con una unión mediante ordenación y fusión.
skew_df.explain("FORMATTED")

In [0]:
display(skew_df)

##### 3.2. Activar AQE y comprobar Data Shuffle, Skew Data y Disk Spilling

In [0]:
spark.conf.set("spark.sql.adaptive.enabled", "true")

In [0]:
spark.conf.get("spark.sql.adaptive.enabled")

In [0]:
spark.conf.get("spark.sql.adaptive.skewJoin.enabled")

In [0]:
spark.conf.get("spark.sql.autoBroadcastJoinThreshold")

In [0]:
spark.conf.get("spark.sql.shuffle.partitions")

In [0]:
# Obtener el importe total de las ventas, agrupado por fecha de venta, para artículos con un precio inferior a 10.
skew_df = spark.sql("""SELECT s_date, sum(s_quantity * i_price) AS total_sales
FROM sales
JOIN items ON s_item_id = i_item_id
WHERE i_price < 10
GROUP BY s_date
ORDER BY total_sales DESC;""") 

In [0]:
# El análisis estático (explain) muestra el plan inicial con una unión mediante ordenación y fusión.
skew_df.explain("FORMATTED")

In [0]:
display(skew_df)

#### Paso 4: Cargar y utilizar los archivos "customers.parquet" y "transactions.parquet"

In [0]:
import pyspark.sql.functions as F
import time

In [0]:
df_transactions = spark.read.parquet("abfss://demo@databricksdeasac.dfs.core.windows.net/raw/data_skew/transactions.parquet/")
df_customers = spark.read.parquet("abfss://demo@databricksdeasac.dfs.core.windows.net/raw/data_skew/customers.parquet/")

In [0]:
df_transactions.printSchema()
df_transactions.show(5, False)

In [0]:
df_customers.printSchema()
df_customers.show(5, False)

##### 4.1. Desactivar AQE y comprobar Data Shuffle, Skew Data y Disk Spilling

In [0]:
spark.conf.set("spark.sql.adaptive.enabled", "false")

In [0]:
spark.conf.get("spark.sql.adaptive.enabled")

In [0]:
spark.conf.get("spark.sql.adaptive.skewJoin.enabled")

In [0]:
spark.conf.get("spark.sql.autoBroadcastJoinThreshold")

In [0]:
spark.conf.get("spark.sql.shuffle.partitions")

In [0]:
# Mostrar los 5 clientes con mayor cantidad de transacciones únicas
(
    df_transactions
    .groupBy("cust_id")
    .agg(F.countDistinct("txn_id").alias("ct"))
    .orderBy(F.desc("ct"))
    .show(5, False)
)

In [0]:
# Capturar las transacciones con la información de los clientes
df_txn_details = (
    df_transactions.join(
        df_customers,
        on = "cust_id",
        how = "inner"
    )
)

In [0]:
# Contar los registros del JOIN y medir el tiempo que se tarda en completar.
start_time = time.time()
display(df_txn_details.count())
print("")
print(f"time_taken: {time.time() - start_time}")

##### 4.2. Activar AQE y comprobar Data Shuffle, Skew Data y Disk Spilling

In [0]:
spark.conf.set("spark.sql.adaptive.enabled", "true")

In [0]:
spark.conf.get("spark.sql.adaptive.enabled")

In [0]:
spark.conf.get("spark.sql.adaptive.skewJoin.enabled")

In [0]:
spark.conf.get("spark.sql.autoBroadcastJoinThreshold")

In [0]:
spark.conf.get("spark.sql.shuffle.partitions")

In [0]:
# Capturar las transacciones con la información de los clientes
df_txn_details = (
    df_transactions.join(
        df_customers,
        on = "cust_id",
        how = "inner"
    )
)

In [0]:
# Contar los registros del JOIN y medir el tiempo que se tarda en completar.
start_time = time.time()
display(df_txn_details.count())
print("")
print(f"time_taken: {time.time() - start_time}")